In [ ]:
"""
Merge atmospheric_spectroscopy.csv with planetary_systems.csv
on planet name, keeping one row per planet from planetary_systems
(the archive's recommended "default" parameter set) joined to
every spectroscopy observation for that planet.
"""

import pandas as pd

# --- Load data ---------------------------------------------------------
spec = pd.read_csv("atmospheric_spectroscopy.csv")

# planetary_systems.csv has leading '#' comment/metadata lines before the
# real header, and a couple of mixed-type columns (hd_name, hip_name)
sys = pd.read_csv("planetary_systems.csv", comment="#", low_memory=False)

# --- Reduce planetary_systems to one row per planet ---------------------
# The archive includes multiple rows per planet (one per reference/solution).
# default_flag == 1 marks the archive's chosen "best" solution for each planet.
sys_default = sys[sys["default_flag"] == 1].copy()

# Guard against any residual duplicate planet names
sys_default = sys_default.drop_duplicates(subset="pl_name")

# --- Merge ---------------------------------------------------------------
# spec key: PL_NAME (uppercase) <-> sys key: pl_name (lowercase)
merged = spec.merge(
    sys_default,
    left_on="PL_NAME",
    right_on="pl_name",
    how="left",          # keep all spectroscopy rows even if no system match
    suffixes=("_spec", "_sys"),
)

print(f"Spectroscopy rows:      {len(spec)}")
print(f"Unique planets (spec):  {spec['PL_NAME'].nunique()}")
print(f"Planetary systems rows: {len(sys_default)}")
print(f"Merged rows:            {len(merged)}")
unmatched = merged[merged["pl_name"].isna()]["PL_NAME"].unique()
print(f"Unmatched planet names: {list(unmatched)}")

# --- Save ------------------------------------------------------------
merged.to_csv("merged_dataset.csv", index=False)
print("Saved merged_dataset.csv")